# 03. Агентний процес під наглядом дослідника

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yevheniikhan/kyiv-forest-agentic-ai/blob/main/notebooks/03_agentic_workflow.ipynb)

Записник показує будову агентної системи на прикладі оцінювання біомаси:

1. **реєстр інструментів**: типізовані функції з описом, лише їх може викликати планувальник;
2. **планувальник**: пропонує послідовність кроків (за замовчуванням сценарний, без ключа API; можна підключити мовну модель);
3. **детерміновані перевірки**: правила дослідника виконуються після кожного кроку незалежно від мовної моделі;
4. **точки рішення дослідника**: план, дані й результат підтверджує людина;
5. **журнал запуску**: кожен крок, аргументи, перевірки й рішення записуються у JSON.

Роль мовної моделі обмежена плануванням і написанням коду. Методику, пороги перевірок і остаточні висновки визначає дослідник.

In [ ]:
# Код репозиторію і бібліотеки
import os, sys
if not os.path.exists('/content/kyiv-forest-agentic-ai'):
    !git clone -q https://github.com/yevheniikhan/kyiv-forest-agentic-ai /content/kyiv-forest-agentic-ai
sys.path.insert(0, '/content/kyiv-forest-agentic-ai/src')
!pip -q install geemap pyproj

import ee
PROJECT = 'gen-lang-client-0200925587'   # замініть на свій проєкт Google Cloud з увімкненим Earth Engine
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()                     # вікно авторизації підтверджує користувач
    ee.Initialize(project=PROJECT)
print('Earth Engine:', ee.String('готово').getInfo())

## 1. Інструменти

In [ ]:
from kyivforest import config as C, ee_data as D, validate as V, services as S
from kyivforest.agent import Registry, CheckResult, run, scripted_planner, llm_planner
import pandas as pd

reg = Registry()

@reg.register('load_samples', 'відбитки GEDI L4A з предикторами Sentinel-1/2 і рельєфу в межах лісу')
def load_samples(state):
    fc = D.training_samples(max_n=4000)
    rows = fc.map(lambda f: f.set({'lon': f.geometry().coordinates().get(0),
                                   'lat': f.geometry().coordinates().get(1)})).getInfo()['features']
    df = V.add_blocks(pd.DataFrame([r['properties'] for r in rows]).dropna())
    state['df'], state['fc'] = df, fc
    state['features'] = [c for c in df.columns if c not in ('agbd', 'lon', 'lat', 'x', 'y', 'block')]
    return {'n': len(df), 'nan_share': float(df.isna().mean().max()),
            'agbd_min': float(df.agbd.min()), 'agbd_max': float(df.agbd.max())}

@reg.register('train_validate', 'навчання RF і крос-валідація; аргумент scheme: random або spatial')
def train_validate(state, scheme='spatial'):
    _, m = V.cross_validate(state['df'], state['features'], kind='rf', scheme=scheme)
    return m

@reg.register('map_and_totals', 'карта AGB у GEE, площа лісу, середня AGB, запас вуглецю')
def map_and_totals(state):
    st = D.area_stats(D.ee_rf_map(state['fc'], state['features'])).getInfo()
    p = S.biomass_to_pools(st['agb_tot_sum'])
    return {'forest_ha': st['ha_sum'], 'mean_agb': st['agb_tot_sum'] / st['ha_sum'],
            'carbon_kt': p['carbon_Mg'] / 1e3, 'co2_kt': p['co2_Mg'] / 1e3}

@reg.register('compare_region', 'порівняння середньої AGB з регіональною щільністю фітомаси 2011 р.')
def compare_region(state):
    return {'ratio_to_region': S.compare_with_region(state['map_and_totals']['mean_agb'])}

print(reg.catalogue())

## 2. Перевірки дослідника

Пороги задано в `config.CHECKS`. Вони відповідають цій демонстрації і не є загальними нормативами.

In [ ]:
K = C.CHECKS
def chk_samples(state, o):
    return [CheckResult('обсяг вибірки', o['n'] >= K['min_samples'], f"n = {o['n']} (мін. {K['min_samples']})"),
            CheckResult('пропуски', o['nan_share'] <= K['max_nan_share'], f"{o['nan_share']:.1%}"),
            CheckResult('діапазон AGBD', K['agbd_range'][0] <= o['agbd_min'] and o['agbd_max'] <= K['agbd_range'][1],
                        f"{o['agbd_min']:.0f}–{o['agbd_max']:.0f} Мг/га")]

def chk_validate(state, o):
    out = [CheckResult('R² більше порогу', o['r2'] >= K['min_r2_spatial'], f"R² = {o['r2']:.2f}")]
    if 'train_validate:random' in state and 'train_validate:spatial' in state:
        gap = state['train_validate:random']['r2'] - state['train_validate:spatial']['r2']
        out.append(CheckResult('розрив random/spatial CV', gap <= K['max_cv_gap'], f'ΔR² = {gap:.2f}'))
    return out

def chk_totals(state, o):
    return [CheckResult('правдоподібна середня AGB', 20 <= o['mean_agb'] <= 400, f"{o['mean_agb']:.1f} Мг/га")]

CHECKS = {'load_samples': chk_samples, 'train_validate': chk_validate, 'map_and_totals': chk_totals}

## 3. Запуск

Під час запуску записник зупиняється і чекає відповіді дослідника. Для демонстрації без зупинок задайте `AUTO = True`.

In [ ]:
AUTO = None   # None: питати дослідника; True: автоматичне підтвердження (лише тест)
state = run('оцінити надземну біомасу й запас вуглецю лісів демонстраційної ділянки',
            reg, CHECKS, planner=scripted_planner, auto=AUTO, log_path='run_log.json')

In [ ]:
import json
log = json.load(open('run_log.json', encoding='utf-8'))
pd.DataFrame([{k: e.get(k) for k in ('t', 'kind', 'tool', 'name', 'passed', 'question', 'approved')} for e in log])

## 4. Планувальник на основі мовної моделі (необов'язково)

Якщо у Secrets Colab є `ANTHROPIC_API_KEY`, план можна отримати від моделі. Відповідь розбирається як JSON, і кожен інструмент звіряється з реєстром до виконання.

In [ ]:
# from google.colab import userdata
# !pip -q install anthropic
# import anthropic
# client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
# plan = llm_planner('оцінити біомасу і перевірити модель просторово', reg, client)
# for s in plan: print(s)